# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
print("Available Record Sets (by @id and name):")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {getattr(record_set, 'name', '<no name>')}")
    print("  Fields:")
    for field in record_set.fields:
        field_label = getattr(field, 'name', getattr(field, 'label', '<no name>'))
        print(f"    - Field @id: {field.id}, name: {field_label}, dtype: {getattr(field, 'data_type', '<no dtype>')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record sets by @id
record_set_ids = [r.id for r in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting records for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")

# For further exploration, pick the first record set if any
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nPreview for record set {main_record_set_id} data:")
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets detected in the metadata.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: pick numeric and categorical fields from the previous overview
import numpy as np

# Choose variables for demonstration. Replace with @id for actual deployment if known.
if record_set_ids:
    df = dataframes[main_record_set_id]

    # Guess a numeric field (find one with float or int dtype or likely protein abundance)
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or 'abundance' in col.lower() or 'coverage' in col.lower() or 'count' in col.lower()]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        print('No obvious numeric field found. Skipping EDA.')
        numeric_field = None

    # Guess a group-by field (e.g. 'description' or 'accession')
    possible_group_fields = [col for col in df.columns if 'desc' in col.lower() or 'accession' in col.lower() or 'sample' in col.lower()]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        print(f"Using grouping field: {group_field}")
    else:
        group_field = None

    # Continue EDA if a numeric field was found
    if numeric_field:
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by group_field
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print('No numeric field found, skipping filtering and normalization.')
else:
    print('No record sets detected; skipping EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize if numeric data exists
if record_set_ids and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If group_field exists and is categorical, plot its means
    if group_field and group_field in df.columns:
        plt.figure(figsize=(9,4))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).dropna()
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.xticks(rotation=90)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print('No numeric field to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.